In [1]:
import os
import pandas as pd
from datetime import datetime
import pytz
import unicodedata
from rapidfuzz import process, fuzz

# === Date & Paths ===
eastern = pytz.timezone("US/Eastern")
today = datetime.now(eastern).date()
today_str = today.isoformat()

paths = {
    "spreads": f"../data/novig-odds/novig_spreads_{today_str}.csv",
    "pitchers": f"../data/starting-pitchers/starting_pitchers_{today_str}.csv",
    "player_pitching": f"../data/player_pitching/player_pitching_{today_str}.csv",
    "standings": f"../data/standings/standings_{today_str}.csv",
    "team_batting": f"../data/team_batting/team_batting_{today_str}.csv",
    "team_pitching": f"../data/team_pitching/team_pitching_{today_str}.csv"
}

# === Load Data ===
for label, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing required file: {path}")

spreads_df = pd.read_csv(paths["spreads"])
pitchers_df = pd.read_csv(paths["pitchers"])
pitching_df = pd.read_csv(paths["player_pitching"])
standings_df = pd.read_csv(paths["standings"])
batting_df = pd.read_csv(paths["team_batting"])
team_pitching_df = pd.read_csv(paths["team_pitching"])

# === Normalize
def normalize_str(s):
    if not isinstance(s, str): return ""
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('utf-8').strip().lower()

# === Clean Names
pitching_df['clean_name'] = pitching_df['Player'].apply(normalize_str)

# === Team Abbr Map
team_name_map = {
    "ATL": "Atlanta Braves", "PHI": "Philadelphia Phillies", "OAK": "Athletics",
    "TOR": "Toronto Blue Jays", "TB": "Tampa Bay Rays", "HOU": "Houston Astros",
    "WAS": "Washington Nationals", "SEA": "Seattle Mariners", "CIN": "Cincinnati Reds",
    "CHC": "Chicago Cubs", "MIL": "Milwaukee Brewers", "CWS": "Chicago White Sox",
    "BAL": "Baltimore Orioles", "LAA": "Los Angeles Angels", "CLE": "Cleveland Guardians",
    "COL": "Colorado Rockies", "NYM": "New York Mets", "SF": "San Francisco Giants",
    "MIA": "Miami Marlins", "BOS": "Boston Red Sox", "STL": "St. Louis Cardinals",
    "TEX": "Texas Rangers", "DET": "Detroit Tigers", "KC": "Kansas City Royals",
    "AZ": "Arizona Diamondbacks", "PIT": "Pittsburgh Pirates", "SD": "San Diego Padres",
    "MIN": "Minnesota Twins", "NYY": "New York Yankees", "LAD": "Los Angeles Dodgers"
}

# === Pitcher Matching
def get_pitcher(team_abbr):
    full_name = team_name_map.get(team_abbr.upper(), None)
    if not full_name:
        return "Unknown"
    home_match = pitchers_df[pitchers_df['Home Team'].str.lower().str.strip() == full_name.lower()]
    away_match = pitchers_df[pitchers_df['Away Team'].str.lower().str.strip() == full_name.lower()]
    if not home_match.empty:
        return home_match['Home Starter'].values[0]
    elif not away_match.empty:
        return away_match['Away Starter'].values[0]
    return "TBD"

spreads_df['fav_pitcher'] = spreads_df['fav_team'].apply(get_pitcher)
spreads_df['dog_pitcher'] = spreads_df['dog_team'].apply(get_pitcher)

# === Fuzzy Match Pitcher Stats
def fuzzy_match_pitcher(name, choices, threshold=80):
    match = process.extractOne(normalize_str(name), choices, scorer=fuzz.ratio, score_cutoff=threshold)
    return match[0] if match else None

def get_pitcher_stats(player):
    norm_name = normalize_str(player)
    row = pitching_df[pitching_df['clean_name'] == norm_name]
    if row.empty:
        match_name = fuzzy_match_pitcher(player, pitching_df['clean_name'].tolist())
        if match_name:
            row = pitching_df[pitching_df['clean_name'] == match_name]
    if row.empty:
        return pd.Series([None]*5)
    row = row.iloc[0]
    return pd.Series([row.get("ERA"), row.get("FIP"), row.get("WHIP"), row.get("SO9"), row.get("BB9"), row.get("ERA+")])

spreads_df[['fav_pitcher_era', 'fav_pitcher_fip', 'fav_pitcher_whip', 'fav_pitcher_so9', 'fav_pitcher_bb9', "fav_pitcher_era+"]] = spreads_df['fav_pitcher'].apply(get_pitcher_stats)
spreads_df[['dog_pitcher_era', 'dog_pitcher_fip', 'dog_pitcher_whip', 'dog_pitcher_so9', 'dog_pitcher_bb9', "dog_pitcher_era+"]] = spreads_df['dog_pitcher'].apply(get_pitcher_stats)

# === Empty Score Columns
spreads_df['fav_score'] = None
spreads_df['dog_score'] = None
spreads_df['home_team'] = spreads_df.apply(lambda row: row['fav_team'] if row['home_favorite'] == 1 else row['dog_team'], axis=1)
spreads_df['away_team'] = spreads_df.apply(lambda row: row['dog_team'] if row['home_favorite'] == 1 else row['fav_team'], axis=1)
spreads_df['home_score'] = None
spreads_df['away_score'] = None

# === Normalize and Join Team Stats
for df in [standings_df, batting_df, team_pitching_df]:
    df['team_clean'] = df['Tm'].apply(normalize_str)

def get_team_stat(team_abbr, df, column):
    full = team_name_map.get(team_abbr.upper(), "")
    clean = normalize_str(full)
    row = df[df['team_clean'] == clean]
    return row[column].values[0] if not row.empty else None

# Add team stats
spreads_df['fav_win_pct'] = spreads_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, 'W-L%'))
spreads_df['dog_win_pct'] = spreads_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, 'W-L%'))
spreads_df['fav_ba'] = spreads_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, 'BA'))
spreads_df['dog_ba'] = spreads_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, 'BA'))
spreads_df['fav_era'] = spreads_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, team_pitching_df, 'ERA'))
spreads_df['dog_era'] = spreads_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, team_pitching_df, 'ERA'))

# Ensure lowercase headers
standings_df.columns = [col.lower() for col in standings_df.columns]


ordered_cols = [
    'game_time_est', 'fav_team', 'fav_score', 'dog_team', 'dog_score',
    'home_team', 'home_score', 'away_team', 'away_score',
    'fav_line', 'dog_line', 'fav_price', 'dog_price',
    'fav_price_american', 'dog_price_american', 'home_favorite', 'market',
    'fav_pitcher', 'dog_pitcher',
    'fav_pitcher_era', 'dog_pitcher_era',
    'fav_pitcher_fip', 'dog_pitcher_fip',
    'fav_pitcher_whip', 'dog_pitcher_whip',
    'fav_pitcher_so9', 'dog_pitcher_so9',
    'fav_pitcher_bb9', 'dog_pitcher_bb9',
    'fav_pitcher_era+', 'dog_pitcher_era+',
    'fav_win_pct', 'dog_win_pct',
    'fav_ba', 'dog_ba',
    'fav_era', 'dog_era'
]

spreads_df = spreads_df[ordered_cols]

# === Show Result
print("✅ Final enriched training set (WAR excluded):")
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
display(spreads_df)

# === Save Final DataFrame to CSV ===
output_path = os.path.join("training-set", f"training_set_{today_str}.csv")
os.makedirs(os.path.dirname(output_path), exist_ok=True)
spreads_df.to_csv(output_path, index=False)
print(f"✅ Enriched spreads saved to {output_path}")

✅ Final enriched training set (WAR excluded):


,game_time_est,fav_team,fav_score,dog_team,dog_score,home_team,home_score,away_team,away_score,fav_line,dog_line,fav_price,dog_price,fav_price_american,dog_price_american,home_favorite,market,fav_pitcher,dog_pitcher,fav_pitcher_era,dog_pitcher_era,fav_pitcher_fip,dog_pitcher_fip,fav_pitcher_whip,dog_pitcher_whip,fav_pitcher_so9,dog_pitcher_so9,fav_pitcher_bb9,dog_pitcher_bb9,fav_pitcher_era+,dog_pitcher_era+,fav_win_pct,dog_win_pct,fav_ba,dog_ba,fav_era,dog_era
0,2025-05-30T19:10:00-04:00,NYM,None,COL,None,COL,None,NYM,None,-1.5,1.5,0.581,0.424,-139,136,0,NYM -1.5,David Peterson,Kyle Freeland,2.79,5.86,3.40,3.42,1.276,1.681,8.4,7.2,3.4,2.1,137.0,79.0,0.607,0.161,.245,.218,2.87,5.55
1,2025-05-30T21:40:00-04:00,ARI,None,WAS,None,ARI,None,WAS,None,-1.5,1.5,0.514,0.512,-106,-105,1,ARI -1.5,Unknown,Jake Irvin,NaN,3.42,NaN,4.48,NaN,1.098,NaN,6.2,NaN,2.5,NaN,116.0,NaN,0.464,.244,.242,4.00,4.98
2,2025-05-30T21:40:00-04:00,SD,None,PIT,None,PIT,None,SD,None,-1.5,1.5,0.459,0.567,118,-131,0,SD -1.5,Nick Pivetta,Mitch Keller,2.72,3.66,3.24,3.16,1.012,1.297,10.1,7.7,2.9,2.4,148.0,115.0,0.574,0.368,.253,.226,3.61,3.99
3,2025-05-30T22:10:00-04:00,NYY,None,LAD,None,LAD,None,NYY,None,-1.5,1.5,0.419,0.596,139,-148,0,LAD +1.5,Max Fried,Tony Gonsolin,1.29,4.68,2.58,4.48,0.929,1.440,8.6,10.1,2.1,4.3,305.0,84.0,0.636,0.607,.259,.263,3.25,4.09
4,2025-05-30T20:10:00-04:00,DET,None,KAN,None,KAN,None,DET,None,-1.5,1.5,0.415,0.618,141,-162,0,KAN +1.5,Casey Mize,Unknown,2.45,NaN,3.90,NaN,1.007,NaN,7.7,NaN,1.9,NaN,160.0,NaN,0.649,NaN,.252,.244,3.22,4.00
5,2025-05-30T19:10:00-04:00,SF,None,MIA,None,MIA,None,SF,None,-1.5,1.5,0.500,0.513,100,-105,0,MIA +1.5,Kyle Harrison,Cal Quantrill,3.86,6.09,4.47,4.52,1.071,1.489,10.6,6.3,2.9,3.0,104.0,72.0,0.554,0.407,.231,.251,3.22,5.29
6,2025-05-30T20:05:00-04:00,STL,None,TEX,None,TEX,None,STL,None,-1.5,1.5,0.439,0.588,128,-143,0,TEX +1.5,Matthew Liberatore,Jack Leiter,2.73,4.17,2.52,4.49,1.011,1.244,7.7,6.6,1.2,4.6,151.0,91.0,0.571,0.474,.262,.220,3.71,3.19
7,2025-05-30T19:10:00-04:00,CLE,None,LAA,None,CLE,None,LAA,None,-1.5,1.5,0.403,0.615,148,-160,1,CLE -1.5,Luis L. Ortiz,José Soriano,4.73,3.73,4.15,3.68,1.425,1.532,10.0,6.8,4.7,4.3,84.0,109.0,0.545,0.455,.233,.224,4.12,4.84
8,2025-05-30T19:07:00-04:00,TOR,None,OAK,None,TOR,None,OAK,None,-1.5,1.5,0.455,0.566,120,-130,1,TOR -1.5,Chris Bassitt,Jeffrey Springs,3.38,3.97,3.24,4.64,1.321,1.220,9.2,7.2,1.9,3.5,121.0,104.0,0.500,0.404,.248,.253,3.93,5.52
9,2025-05-30T20:10:00-04:00,HOU,None,TB,None,HOU,None,TB,None,-1.5,1.5,0.415,0.605,141,-153,1,HOU -1.5,Framber Valdez,Ryan Pepiot,3.39,3.55,3.29,4.43,1.145,1.232,8.3,7.4,3.0,2.7,118.0,108.0,0.536,0.518,.255,.247,3.55,3.57


✅ Enriched spreads saved to training-set/training_set_2025-05-30.csv
